In [1]:
import reasoning_gym

from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM)

from peft import LoraConfig, get_peft_model


class Maze:
    def __init__(self, tokenizer, dataset_name="maze", min_dist=3, max_dist=10, size=2, min_grid_size=5, max_grid_size=5, seed=None):
        """Initialize maze with dataset parameters"""
        self.tokenizer = tokenizer

        self.dataset = list(reasoning_gym.create_dataset(
            dataset_name,
            min_dist=min_dist,
            max_dist=max_dist,
            size=size,
            min_grid_size=min_grid_size,
            max_grid_size=max_grid_size,
            seed=seed
        ))

    def get_maze_world_prompt(self, index):
        """Return only the maze visualization text without the question"""
        question = self.dataset[index]['question']
        end_of_grid_text = question.index("What")

        grid_world_prompt = question[:end_of_grid_text].strip()
        grid_world_prompt += """\nOutput ONLY the steps needed to go from the start position to the goal position. Use ONLY the words (up, down, left, right) to solve the maze.\n<example_output>\nup, right, up, up, left, up\n</example_output>"""

        messages = [{"role": "user", "content": grid_world_prompt}]

        formatted_prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True  # Adds the assistant turn start
        )
        return formatted_prompt

    def get_prompts(self):
        """Returns all prompts in list format"""
        return [self.get_maze_world_prompt(maze_idx) for maze_idx, _ in enumerate(self.dataset)]

# Usage
# Select model

model_id = "hugging-quants/Llama-3.2-3B-Instruct-Q4_K_M-GGUF"
filename = "llama-3.2-3b-instruct-q4_k_m.gguf"

tok = AutoTokenizer.from_pretrained(model_id, gguf_file=filename)
model = AutoModelForCausalLM.from_pretrained(model_id, gguf_file=filename)
# model_name_path = "models/llama_3.2_3b_instruct_q4"

device = "mps" if torch.backends.mps.is_available() else "cpu"
# device = 'cpu'
# Load tokenizer
# tok = AutoTokenizer.from_pretrained(model_name_path, use_fast=True)
tok.pad_token = tok.eos_token

maze = Maze(tokenizer=tok, size=10, min_dist=2, max_dist=5, min_grid_size=4, max_grid_size=7, seed=1)

mazes = maze.get_prompts()
mazes_dicts = [{"prompt": maze_prompt} for maze_prompt in mazes]
dataset = Dataset.from_list(mazes_dicts)

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


Converting and de-quantizing GGUF tensors...:   0%|          | 0/255 [00:00<?, ?it/s]

In [2]:
print(dataset)
print(dataset[0])  # See the first sample
print(dataset.column_names)  # Check column names

Dataset({
    features: ['prompt'],
    num_rows: 10
})
{'prompt': "<bos><start_of_turn>user\nNavigate from '8' (start) to 'w' (goal):\n\n```\nhhhh\nh+8h\nhwhh\nhhhh\n```\nLegend: 'h' = Wall, '+' = Passage\nOutput ONLY the steps needed to go from the start position to the goal position. Use ONLY the words (up, down, left, right) to solve the maze.\n<example_output>\nup, right, up, up, left, up\n</example_output><end_of_turn>\n<start_of_turn>model\n"}
['prompt']


In [3]:
import reasoning_gym

from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM)

from peft import LoraConfig

class Maze:
    def __init__(self, tokenizer, dataset_name="maze", min_dist=3, max_dist=10, size=2, min_grid_size=5, max_grid_size=5, seed=None):
        """Initialize maze with dataset parameters"""
        self.tokenizer = tokenizer

        self.dataset = list(reasoning_gym.create_dataset(
            dataset_name,
            min_dist=min_dist,
            max_dist=max_dist,
            size=size,
            min_grid_size=min_grid_size,
            max_grid_size=max_grid_size,
            seed=seed
        ))

    def get_maze_world_prompt(self, index):
        """Return only the maze visualization text without the question"""
        question = self.dataset[index]['question']
        end_of_grid_text = question.index("What")

        grid_world_prompt = question[:end_of_grid_text].strip()
        grid_world_prompt += """\nOutput ONLY the steps needed to go from the start position to the goal position. Use ONLY the words (up, down, left, right) to solve the maze.\n<example_output>\nup, right, up, up, left, up\n</example_output>"""

        messages = [{"role": "user", "content": grid_world_prompt}]

        formatted_prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True  # Adds the assistant turn start
        )
        return formatted_prompt

    def get_prompts(self):
        """Returns all prompts in list format"""
        return [self.get_maze_world_prompt(maze_idx) for maze_idx, _ in enumerate(self.dataset)]

# Usage
# Select model
model_name_path = "models/gemma_3.1_4B_instruct"

device = "mps" if torch.backends.mps.is_available() else "cpu"
# device = 'cpu'
# Load tokenizer
tok = AutoTokenizer.from_pretrained(model_name_path, use_fast=True)
tok.pad_token = tok.eos_token

maze = Maze(tokenizer=tok, size=10, min_dist=2, max_dist=5, min_grid_size=4, max_grid_size=7, seed=1)

mazes = maze.get_prompts()
mazes_dicts = [{"prompt": maze_prompt} for maze_prompt in mazes]
dataset = Dataset.from_list(mazes_dicts)

# dataset = load_dataset("trl-lib/DeepMath-103K", split="train")
#
# print()

def my_stuff(completions: list[list[dict[str, str]]], **kwargs) -> list[float | None]:
    rewards = []
    for content in completions:
        rewards.append(1)

    return rewards



# Load model
model = AutoModelForCausalLM.from_pretrained(model_name_path)
# model.to(device)

# Memory optimization for MPS
# model.config.use_cache = False
# if hasattr(model, "gradient_checkpointing_enable"):
#     model.gradient_checkpointing_enable()


# LoRA config
lora_cfg = LoraConfig(
r=8, lora_alpha=16, lora_dropout=0.05,
target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
task_type="CAUSAL_LM"
)

training_args = GRPOConfig(
    output_dir="./output",
    beta=0, # if > 0 it would load a second model needed for calculating KL divergence
    num_generations=6, # default is 8
    max_completion_length=128, # default is 256
    per_device_train_batch_size=6, # default is 8
    gradient_checkpointing=True,
    log_completions=True,
    logging_steps=1,
    gradient_accumulation_steps=16,
    num_train_epochs=1,

    # Numerical stability fixes
    # bf16=False,
    # fp16=True,  # Use float32 for MPS stability
    # use_vllm=True,
    # vllm_mode='colocate' # vLLM is built for CUDA, so tweaking will need to be done for this to work on Mac
)
trainer = GRPOTrainer(
    args=training_args,
    model=model,
    reward_funcs=my_stuff,
    train_dataset=dataset,
    peft_config=lora_cfg,

)
# Check where model parameters actually are
print(f"Model device: {next(model.parameters()).device}")

# After creating trainer, you can also check:
print(f"Trainer device: {trainer.args.device}")
# trainer.train()
# # Save LoRA adapter
# model.save_pretrained("output/lora")
# tok.save_pretrained("output/lora")
# print("Done. LoRA adapter saved to out/lora")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Model device: mps:0
Trainer device: mps


In [4]:
# Before training, check what the trainer expects
print(trainer.train_dataset)
print(trainer.train_dataset.column_names)

Dataset({
    features: ['prompt'],
    num_rows: 10
})
['prompt']


In [5]:
inputs = tok(dataset[0]["prompt"], return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=50)
print(tok.decode(output[0]))

<bos><bos><start_of_turn>user
Navigate from '8' (start) to 'w' (goal):

```
hhhh
h+8h
hwhh
hhhh
```
Legend: 'h' = Wall, '+' = Passage
Output ONLY the steps needed to go from the start position to the goal position. Use ONLY the words (up, down, left, right) to solve the maze.
<example_output>
up, right, up, up, left, up
</example_output><end_of_turn>
<start_of_turn>model
right, right, down, down, left, down, right
<end_of_turn>


In [1]:
import time
import reasoning_gym
import torch
from transformers import (AutoTokenizer, AutoModelForCausalLM)

filename = "models/gemma_3.1_4B_instruct"
device = "mps"
print(f"Device: {device}")

start = time.time()
tok = AutoTokenizer.from_pretrained(filename, use_fast=True)
print(f"Tokenizer load: {time.time() - start:.2f}s")
start = time.time()
model = AutoModelForCausalLM.from_pretrained(filename)
print(f"Model load: {time.time() - start:.2f}s")
start = time.time()

# Profile the chat function
def chat(prompt, max_new_tokens=128):
    t0 = time.time()

    messages = [{"role": "user", "content": prompt}]
    input_text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(input_text, return_tensors="pt").to(device)
    print(f"  Tokenization: {time.time() - t0:.3f}s")

    t1 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tok.eos_token_id,
        )
    if device == "mps":
        torch.mps.synchronize()  # Wait for MPS to finish
    print(f"  Generation: {time.time() - t1:.3f}s")

    t2 = time.time()
    response = tok.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"  Decoding: {time.time() - t2:.3f}s")

    print(f"  Total: {time.time() - t0:.3f}s")
    print(f"  Tokens generated: {len(outputs[0]) - inputs['input_ids'].shape[1]}")

    return response

# Test it
maze_prompt = """Navigate from 'a' (start) to 'K' (goal):
```
7777777
7b77b77
77b7bb7
7bbbbb7
7ba7Kb7
7b7bbb7
7777777
```
Legend: '7' = Wall, 'b' = Passage

Output ONLY the steps needed. Use ONLY: up, down, left, right"""

response = chat(maze_prompt)
print(response)

Device: mps
Tokenizer load: 0.45s


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model load: 13.83s
  Tokenization: 0.167s


/Users/gjorgjinoveski/PycharmProjects/llm_fine_tune/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:2532: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on mps, whereas the model is on cpu. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cpu') before running `.generate()`.
  warnings.warn(


RuntimeError: Placeholder storage has not been allocated on MPS device!

THIS IS the hugging-quants/Llama-3.2-3B-Instruct-Q4_K_M-GGUF modle ON CPU!!!



In [6]:
from mlx_lm import load, generate

model, tokenizer = load("mlx-community/Llama-3.2-3B-Instruct-4bit")

prompt="""Navigate from 'a' (start) to 'K' (goal):
```
7777777
7b77b77
77b7bb7
7bbbbb7
7ba7Kb7
7b7bbb7
7777777
```
Legend: '7' = Wall, 'b' = Passage

Output ONLY the steps needed. Use ONLY: up, down, left, right"""

if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
    messages = [{"role": "user", "content": prompt}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

response = generate(model, tokenizer, prompt=prompt, verbose=True)
# print(response)response

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

To navigate from 'a' (start) to 'K' (goal), the steps are:

1. Down
2. Down
3. Down
4. Down
5. Right
6. Right
7. Right
8. Right
9. Right
10. Right
11. Right
12. Right
13. Right
14. Right
15. Right
16. Right
17. Right
18. Right
19. Right
20. Right
21. Right
22. Right
23. Right
24. Right
25. Right
26. Right
27. Right
28. Right
29. Right
30. Right
31. Right
32. Right
33. Right
34. Right
35. Right
36. Right
37. Right
38. Right
39. Right
40. Right
41. Right
42. Right
43. Right
44. Right
45. Right
46. Right
47. Right
48. Right
49. Right
50. Right
51. Right
52. Right
53. Right
54. Right
55. Right
56. Right
57. Right
58. Right
59. Right

Prompt: 122 tokens, 610.525 tokens-per-sec
Generation: 256 tokens, 112.902 tokens-per-sec
Peak memory: 3.615 GB
